# Capítulo 3 — Experimentos Estatísticos e Teste de Significância

**Livro:** começa na página **115** do PDF.

📗 **Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
rng = np.random.default_rng(42)
print("Pronto. pandas", pd.__version__)

## Teste de Permutação · PDF p. 127

#### 🧪 Exercício: Teste de permutação

Tempo na página: A tem 30 sessões ~ normal(60, 20) e B tem 25 ~ normal(75, 20). Faça um teste de permutação com 5.000 embaralhamentos para a diferença de médias e calcule o p-valor bilateral.

In [ ]:
a = rng.normal(60, 20, 30); b = rng.normal(75, 20, 25)
obs = b.mean() - a.mean(); tudo = np.concatenate([a, b])
difs = []
for _ in range(5000):
    p = rng.permutation(tudo); difs.append(p[30:].mean() - p[:30].mean())
print(round(obs, 2), np.mean(np.abs(difs) >= abs(obs)))

## Erros Tipo 1 e Tipo 2 · PDF p. 138

#### 🧪 Exercício: Erro tipo 1 e tipo 2 por simulação

Com α = 5% e teste t para duas amostras de 20: (a) sem efeito real, qual fração dos 2.000 testes rejeita H₀? (b) com diferença real de 0,5 desvio, qual fração não rejeita (erro tipo 2)?

In [ ]:
def rejeita(efeito):
    return stats.ttest_ind(rng.normal(0, 1, 20), rng.normal(efeito, 1, 20)).pvalue < 0.05
print(np.mean([rejeita(0) for _ in range(2000)]), 1 - np.mean([rejeita(0.5) for _ in range(2000)]))

## Testes t · PDF p. 140

#### 🧪 Exercício: Teste t de Welch

Compare a pressão (`bp`) de homens e mulheres nos dados de diabetes com `stats.ttest_ind(..., equal_var=False)`.

In [ ]:
from sklearn.datasets import load_diabetes
df = load_diabetes(as_frame=True, scaled=False).frame
g1, g2 = [g["bp"] for _, g in df.groupby("sex")]
print(stats.ttest_ind(g1, g2, equal_var=False))

## Testagem Múltipla · PDF p. 143

#### 🧪 Exercício: Correção para testes múltiplos

Gere 100 p-valores de testes sem efeito (uniformes) e 5 de efeitos reais (`rng.uniform(0, 0.001, 5)`). Conte quantos passam em α = 5% sem correção, com Bonferroni e com Benjamini-Hochberg (`statsmodels.stats.multitest.multipletests`).

In [ ]:
from statsmodels.stats.multitest import multipletests
p = np.concatenate([rng.uniform(0, 1, 100), rng.uniform(0, 0.001, 5)])
print((p < 0.05).sum(), multipletests(p, 0.05, "bonferroni")[0].sum(), multipletests(p, 0.05, "fdr_bh")[0].sum())

## Estatística F · PDF p. 153

#### 🧪 Exercício: ANOVA de um fator

Simule o tempo de permanência em 4 versões de página (médias 60, 62, 61 e 70, desvio 10, 25 sessões cada) e rode `stats.f_oneway`. Depois faça a mesma ANOVA com `smf.ols` + `sm.stats.anova_lm`.

In [ ]:
medias = [60, 62, 61, 70]
df = pd.DataFrame([(f"p{i}", t) for i, m in enumerate(medias) for t in rng.normal(m, 10, 25)], columns=["pagina", "tempo"])
print(stats.f_oneway(*[g["tempo"] for _, g in df.groupby("pagina")]))
print(sm.stats.anova_lm(smf.ols("tempo ~ C(pagina)", df).fit()))

## ANOVA Bidirecional · PDF p. 155

#### 🧪 Exercício: ANOVA com dois fatores

Acrescente um segundo fator, `dispositivo` (celular/computador, +5 s no computador), e ajuste `tempo ~ C(pagina) * C(dispositivo)`. Mostre a tabela ANOVA.

In [ ]:
linhas = []
for i, m in enumerate([60, 62, 61, 70]):
    for disp, extra in [("celular", 0), ("computador", 5)]:
        linhas += [(f"p{i}", disp, t) for t in rng.normal(m + extra, 10, 15)]
df = pd.DataFrame(linhas, columns=["pagina", "dispositivo", "tempo"])
print(sm.stats.anova_lm(smf.ols("tempo ~ C(pagina) * C(dispositivo)", df).fit(), typ=2).round(3))

## Teste de Qui Quadrado: Uma Abordagem à Reamostra · PDF p. 157

#### 🧪 Exercício: Qui-quadrado por reamostragem e pela teoria

Três manchetes tiveram cliques/visualizações 14/1000, 8/1000 e 12/1000. Faça o teste qui-quadrado com `stats.chi2_contingency` e por reamostragem (5.000 simulações sob H₀).

In [ ]:
cliques = np.array([14, 8, 12]); n = 1000
tab = np.array([cliques, n - cliques])
chi2, p, *_ = stats.chi2_contingency(tab); print(round(chi2, 3), round(p, 4))
taxa = cliques.sum() / (3 * n)
def chi2_de(c):
    esp = np.array([[taxa * n] * 3, [(1 - taxa) * n] * 3]); obs = np.array([c, n - c])
    return ((obs - esp) ** 2 / esp).sum()
obs = chi2_de(cliques)
sims = [chi2_de(rng.binomial(n, taxa, 3)) for _ in range(5000)]
print(round(obs, 3), np.mean(np.array(sims) >= obs))

## Teste Exato de Fisher · PDF p. 160

#### 🧪 Exercício: Fisher para amostras pequenas

Num teste com poucos dados, A converteu 1 de 10 e B 7 de 12. Rode o teste exato de Fisher e compare com o qui-quadrado (que não é confiável aqui).

In [ ]:
tab = [[1, 9], [7, 5]]
print(stats.fisher_exact(tab), stats.chi2_contingency(tab)[1])

## Algoritmo de Bandido Multibraços · PDF p. 164

#### 🧪 Exercício: Bandido de vários braços (epsilon-greedy)

Três ofertas convertem 4%, 5% e 7% (você não sabe). Simule 10.000 visitantes com epsilon-greedy (ε = 0,1): explora ao acaso com prob. ε, senão mostra a de maior taxa observada. Quantas vezes cada oferta foi mostrada?

In [ ]:
taxas = np.array([0.04, 0.05, 0.07]); mostra = np.zeros(3); conv = np.zeros(3)
for _ in range(10_000):
    braco = rng.integers(3) if rng.random() < 0.1 or mostra.min() == 0 else int(np.argmax(conv / mostra))
    mostra[braco] += 1; conv[braco] += rng.random() < taxas[braco]
print(mostra.astype(int), (conv / mostra).round(4))

## Tamanho da Amostra · PDF p. 171

#### 🧪 Exercício: Potência e tamanho de amostra

Quantos visitantes por grupo são necessários para detectar uma mudança de conversão de 10% para 11% com α = 5% e potência de 80%? Use `NormalIndPower` e `proportion_effectsize` do statsmodels.

In [ ]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize
ef = proportion_effectsize(0.11, 0.10)
print(round(NormalIndPower().solve_power(ef, alpha=0.05, power=0.8)))